# Mining Distance Metrics Workshop Lab

## Scenario

You are part of a mining operations analytics team reviewing equipment sensor readings from different areas of a mine site.  
Each observation represents a machine reading with four measured features:

- **temperature**
- **vibration**
- **pressure**
- **humidity**

The operations team wants to answer practical questions such as:

- Which machines appear to be operating in a similar way?
- Which readings look quite different and may need review?
- How does the definition of "similar" change depending on the metric we choose?

In this activity, you will use distance metrics to compare sensor readings and connect those results to simple machine learning tasks.

## Key ideas

This lab introduces the following ideas:

- an **observation** is one row of data
- a **feature** is one measured column
- a **distance metric** gives a numerical way to compare two observations
- different metrics capture different meanings of similarity
- distance-based comparisons are often used in **anomaly review**, **clustering**, and **nearest-neighbour** style analysis

## Learning goals

By the end of the lab, you should be able to:

1. load a sensor dataset into a notebook
2. inspect rows, columns, and summary statistics
3. compare two observations using Euclidean distance, Manhattan distance, and cosine similarity
4. explain what each metric is emphasising
5. create a distance matrix and identify similar rows
6. connect these ideas to mining use cases


## How to use this notebook

Follow the notebook from top to bottom.

### Running cells

- Click on a cell.
- Press **Shift + Enter** to run it.
- Wait for the output to appear before moving to the next cell.

### How to work through the lab

1. Read the explanation above each code cell.
2. Run the code cell.
3. Look at the output and compare it to the explanation.
4. Complete the checkpoint tasks where prompted.
5. Use the worked solution cells only after attempting each task yourself.

### If something goes wrong

- Re-run the current cell.
- Check whether you already ran the earlier setup cells.
- Make sure the dataset path is correct.
- If needed, restart the notebook kernel/runtime and run all cells again from the top.


## 1) Set up the environment


In [ ]:
# Import the libraries used in this workshop.
# Each library gives us a set of tools for a different kind of task.

import pandas as pd               # Work with tabular data such as CSV files
import numpy as np                # Perform numerical calculations with arrays
import matplotlib.pyplot as plt   # Create charts and visualisations

# Import distance and similarity functions from scikit-learn.
from sklearn.metrics.pairwise import euclidean_distances, cosine_similarity


### Why we do this

Before we can analyse the data, we need to load the Python tools used in the notebook.

- **pandas** handles data tables
- **numpy** supports calculations
- **matplotlib** draws charts
- **scikit-learn** provides distance and similarity utilities


## 2) Load the dataset


In [ ]:
# Load the mining sensor dataset from the repository.
# The relative path works when the notebook is kept inside the notebooks/ folder.

DATA_URL = "https://raw.githubusercontent.com/albert-axi/mining-distance-metrics-workshop/main/data/sample_mining_sensors.csv"
df = pd.read_csv(DATA_URL)

# Display the first few rows so we can see what the dataset looks like.
df.head()


### What this cell is doing

This code reads a CSV file into a pandas **DataFrame**, which behaves like a spreadsheet table in Python.

We then display the first five rows so we can check:

- what the columns are called
- whether the data loaded correctly
- what each record looks like


## 3) Inspect the data


In [ ]:
# Check the number of rows and columns in the dataset.
# The result is returned as: (rows, columns)

df.shape


In [ ]:
# Show the column names.
# This helps us identify which fields are useful for distance calculations.

df.columns.tolist()


In [ ]:
# Summarise the numeric features.
# This gives us count, average, spread, minimum, and maximum values.

df.describe(numeric_only=True)


In [ ]:
# Create histograms for the four sensor features.
# A histogram helps us see how values are distributed.

df[["temperature", "vibration", "pressure", "humidity"]].hist(figsize=(10, 6))
plt.tight_layout()
plt.show()


### Why inspection matters

We inspect the data before comparing records because distance metrics depend on the values in each feature.

Questions to think about:

- Are some features much larger in scale than others?
- Are some readings unusually high or low?
- Do the features appear tightly grouped or widely spread?

These questions matter because they affect how similarity is measured.


## 4) Select two sensor readings to compare


In [ ]:
# Choose the features we want to compare.
# These are the numerical measurements that represent each sensor reading.

features = ["temperature", "vibration", "pressure", "humidity"]

# Select row 0 and row 1 from the dataset.
# .to_numpy() converts the selected values into NumPy arrays,
# which are easier to use in mathematical calculations.

sensor_a = df.loc[0, features].to_numpy()
sensor_b = df.loc[1, features].to_numpy()

print("Sensor A values:")
print(sensor_a)
print()
print("Sensor B values:")
print(sensor_b)


In [ ]:
# Show the same two records in table form.
# This is often easier to read than looking at raw arrays.

df.loc[[0, 1], ["reading_id", "site_zone", "equipment_type"] + features]


### Why we start with two rows

Starting with only two rows makes the ideas easier to understand.

Once you know how two observations are compared, you can extend the same logic to all rows in the dataset.


## 5) Euclidean distance


**Plain-language idea:** Euclidean distance is the straight-line distance between two points.

**Mining interpretation:** Use this when you care about the overall size of the difference across all sensor features.


In [ ]:
# Calculate Euclidean distance between the two selected sensor readings.
#
# Step by step:
# 1. sensor_a - sensor_b calculates the feature-by-feature differences.
# 2. np.linalg.norm(...) combines those differences into a single distance value.
#
# Interpretation:
# - smaller value -> the two readings are more similar
# - larger value  -> the two readings are more different

euclid = np.linalg.norm(sensor_a - sensor_b)
print("Euclidean distance:", round(float(euclid), 4))


## 6) Manhattan distance


**Plain-language idea:** Manhattan distance adds the absolute difference across each feature.

**Mining interpretation:** This can be useful when you want to treat each feature difference as a separate step and then total them.


In [ ]:
# Calculate Manhattan distance.
#
# Step by step:
# 1. sensor_a - sensor_b finds the difference for each feature.
# 2. np.abs(...) converts every difference to a positive value.
# 3. np.sum(...) adds those values together.

manhattan = np.sum(np.abs(sensor_a - sensor_b))
print("Manhattan distance:", round(float(manhattan), 4))


## 7) Cosine similarity


**Plain-language idea:** Cosine similarity compares the direction of two vectors rather than their overall size.

**Mining interpretation:** This is useful when the pattern across features matters more than the raw magnitude.


In [ ]:
# Calculate cosine similarity.
#
# cosine_similarity expects 2D inputs, so each sensor vector is wrapped in [ ].
#
# Interpretation:
# - closer to 1  -> very similar direction/pattern
# - closer to 0  -> weak similarity
# - closer to -1 -> opposite direction

cos = cosine_similarity([sensor_a], [sensor_b])[0, 0]
print("Cosine similarity:", round(float(cos), 4))


### Checkpoint 1

In your own words, answer these questions:

1. Which metric says the two readings are most similar?
2. Which metric focuses on absolute size of difference?
3. Which metric focuses more on pattern than magnitude?

Add your answers in a new Markdown cell before moving on.


## 8) Build a distance matrix


In [ ]:
# Convert the selected feature columns into a numeric matrix.
# Each row is one observation and each column is one feature.

X = df[features].to_numpy()

# Compute the Euclidean distance between every pair of rows.
# The result is a square matrix:
# - row i compared with row j
# - column j compared with row i

dist_matrix = euclidean_distances(X)

print("Distance matrix shape:", dist_matrix.shape)
pd.DataFrame(dist_matrix).head()


### What to notice

- The matrix is square because every row is compared with every other row.
- The diagonal is 0 because each row has zero distance from itself.
- Smaller off-diagonal values indicate more similar observations.


## 9) Find the most similar pair in a subset


In [ ]:
# Use only the first 25 rows so the example stays easy to inspect.
subset = df.head(25)[features].to_numpy()

# Calculate Euclidean distances within the subset.
d = euclidean_distances(subset)

# Ignore self-comparisons by replacing the diagonal with infinity.
# This prevents a row from being selected as the closest match to itself.
np.fill_diagonal(d, np.inf)

# Find the smallest remaining value in the matrix.
i, j = np.unravel_index(np.argmin(d), d.shape)

print(f"Closest pair in the first 25 rows: row {i} and row {j}")
print(f"Euclidean distance: {d[i, j]:.4f}")


In [ ]:
# Display the two most similar rows that were found.

df.loc[[i, j], ["reading_id", "site_zone", "equipment_type"] + features]


### Why we replace the diagonal

Every row is identical to itself, so the diagonal entries are all zero.

If we left those zeros in place, the algorithm would always say a row is most similar to itself, which is not helpful when we are trying to compare different readings.


## 10) Visualise two features


In [ ]:
# Plot temperature against vibration.
# Each point represents one sensor reading.

plt.figure(figsize=(7, 5))
plt.scatter(df["temperature"], df["vibration"])
plt.xlabel("Temperature")
plt.ylabel("Vibration")
plt.title("Mining sensor readings: Temperature vs Vibration")
plt.show()


### What this chart helps us do

The scatter plot gives a visual way to think about similarity.

You can look for:

- points that sit close together
- points that sit far away from the main group
- possible unusual readings that may deserve investigation


## 11) Guided practice task


### Task

Using the **first 10 rows** of the dataset:

1. compute the cosine similarity matrix
2. ignore the diagonal so a row is not matched with itself
3. find the pair with the **highest** cosine similarity
4. plot the feature values for that pair side by side

Try to complete the cell below before opening the worked solution.


In [ ]:
# Guided practice: complete the TODO steps.

subset = df.head(10)[features].to_numpy()

# Step 1: compare every row with every other row using cosine similarity
cos_m = cosine_similarity(subset)

# Step 2: ignore self-comparisons
# TODO: replace the diagonal values with -np.inf

# Step 3: find the indices of the highest similarity value
# TODO: use np.unravel_index and np.argmax

# Step 4: display the row numbers and similarity score
# TODO: print the pair and the score

# Step 5: plot the two rows on a bar chart
# TODO: create a bar chart that compares the feature values


### Checkpoint 2

Before you reveal the solution, explain why the cosine similarity task searches for the **largest** value rather than the smallest one.


## 12) Worked solution


In [ ]:
# Worked solution for the guided practice task.

subset = df.head(10)[features].to_numpy()
cos_m = cosine_similarity(subset)

# Ignore self-comparisons by assigning a very small placeholder value.
np.fill_diagonal(cos_m, -np.inf)

# Find the location of the highest cosine similarity.
i, j = np.unravel_index(np.argmax(cos_m), cos_m.shape)

print(f"Most similar pair by cosine similarity in the first 10 rows: row {i} and row {j}")
print(f"Cosine similarity: {cos_m[i, j]:.4f}")

# Plot the feature values side by side.
x = np.arange(len(features))
width = 0.35

plt.figure(figsize=(9, 5))
plt.bar(x - width/2, subset[i], width=width, label=f"Row {i}")
plt.bar(x + width/2, subset[j], width=width, label=f"Row {j}")
plt.xticks(x, features)
plt.ylabel("Feature value")
plt.title("Most similar pair by cosine similarity")
plt.legend()
plt.show()


## 13) Optional extension: simple clustering


In [ ]:
# K-Means is a clustering algorithm that groups observations into clusters.
# It uses distances to decide which rows are most alike.

from sklearn.cluster import KMeans

X = df[features].to_numpy()

# Create a model with 3 clusters.
# random_state=42 makes the result repeatable.
# n_init='auto' controls how many starting configurations are tested.
kmeans = KMeans(n_clusters=3, random_state=42, n_init="auto")

# fit_predict trains the model and returns the assigned cluster for each row.
clusters = kmeans.fit_predict(X)

df["cluster"] = clusters
df[["reading_id", "site_zone", "equipment_type", "cluster"] + features].head()


In [ ]:
# Visualise the clusters using temperature and vibration.

plt.figure(figsize=(7, 5))
plt.scatter(df["temperature"], df["vibration"], c=df["cluster"])
plt.xlabel("Temperature")
plt.ylabel("Vibration")
plt.title("K-Means clusters on mining sensor readings")
plt.show()


### Why this extension matters

Distance metrics are not only mathematical definitions.

They are used inside practical machine learning workflows such as:

- clustering related observations
- finding nearest neighbours
- highlighting unusual behaviour


## 14) Reflection questions


Write your answers in a new Markdown cell.

1. Why might Euclidean distance and cosine similarity rank similarity differently?
2. In a mining setting, when would overall magnitude matter more than pattern?
3. When could cosine similarity be a better choice than Euclidean distance?
4. Why can feature scale affect distance-based analysis?


## Wrap-up

You have completed a workshop lab on distance metrics using mining-style sensor data.

### What you covered

- loading and inspecting data
- comparing observations with Euclidean distance
- comparing observations with Manhattan distance
- comparing observations with cosine similarity
- building a distance matrix
- identifying similar records
- connecting distance metrics to clustering and practical mining analysis

### Practical connection to mining

The same ideas can support tasks such as:

- identifying similar equipment behaviour
- spotting unusual sensor patterns
- grouping operating states
- supporting maintenance review and investigation
